In [2]:

def project_point_with_distortion(xyz_world, R, t, f, cx, cy, k):
    """Returns (u, v, z_cam) for a COLMAP 3D point."""
    X_cam = R @ xyz_world + t
    z = X_cam[2]
    if z <= 0:
        return None
    xn = X_cam[0] / z
    yn = X_cam[1] / z
    r2 = xn**2 + yn**2
    factor = 1 + k * r2
    u = f * xn * factor + cx
    v = f * yn * factor + cy
    return u, v, z


In [4]:
import numpy as np
from viser.extras.colmap import (
    read_cameras_binary,
    read_images_binary,
    read_points3d_binary,
)
np.set_printoptions(precision=3)
path="model/"

In [5]:
cameras=read_cameras_binary(path+"sparse/0/cameras.bin")
images=read_images_binary(path+"sparse/0/images.bin")
points3D=read_points3d_binary(path+"sparse/0/points3D.bin")

In [6]:
points2D=images[1].xys[images[1].point3D_ids!=-1]
points3D_ids=images[1].point3D_ids[images[1].point3D_ids!=-1]
points3D_coord=[points3D[i].xyz for i in points3D_ids]


In [7]:
params=cameras[1].params
intrinsic=np.array([np.zeros(3) for i in range(3)])
intrinsic[2,2]=1
intrinsic[0,0]=params[0]
intrinsic[1,1]=params[0]
intrinsic[0,2]=params[1]
intrinsic[1,2]=params[2]

In [8]:
ROT=np.array(images[1].qvec2rotmat())
T=np.array(images[1].tvec)
extrinsic=np.c_[ROT,T]
p1=np.r_[np.array(points3D_coord[11]).T,np.array([1])]

In [24]:
result=intrinsic.dot(extrinsic).dot(p1)
result=(result/result[2]).astype(int)
print(result)
print(points2D[10])

[2080 1286    1]


[2888.95947266 1362.44189453]


In [23]:
project_point_with_distortion(p1[:3].T, ROT, T, params[0], params[1], params[2], params[3])

(np.float64(2080.967181755665), np.float64(1286.726951960478), np.float64(152.7005146819986))